In [ ]:
from statsbombpy import sb
import pandas as pd

matches = sb.matches(competition_id=11, season_id=27)
match_ids = matches['match_id'].tolist()

all_events = []
for match_id in match_ids:
    events = sb.events(match_id=match_id)
    all_events.append(events)

events_df = pd.concat(all_events, ignore_index=True)
print(f"Total events loaded: {len(events_df)}")

In [ ]:
import os
save_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data', 'la_liga_1516_events.parquet')
events_df.to_parquet(save_path, index=False)
print(f"Saved to: {save_path}")
print(f"File exists: {os.path.exists(save_path)}")

In [ ]:
# StatsBomb coordinates are in yards. Pitch dimensions: 120 x 80 yards.
# Goal center: [120, 40]. All distance calculations produce values in yards.

shots = events_df[events_df['type']  == 'Shot'].copy()
print(f"Total shots: {len(shots)}")
print(shots[['player', 'location', 'shot_statsbomb_xg']].head(10))

Total shots: 9168
                            player       location  shot_statsbomb_xg
2356  Jefferson Andrés Lerma Solís  [112.5, 42.1]           0.100429
2357        Sergio Gontán Gallardo   [90.9, 39.9]           0.023138
2358       Adrián González Morales  [103.1, 52.5]           0.034535
2359          Borja González Tomás  [116.6, 38.6]           0.394774
2360                  Nabil Ghilas  [112.4, 35.8]           0.077194
2361                  Nabil Ghilas  [102.6, 39.7]           0.072501
2362                  Takashi Inui  [102.6, 30.4]           0.049790
2363       Adrián González Morales  [109.9, 32.8]           0.151259
2364                  Nabil Ghilas   [99.1, 38.2]           0.074698
2365   José Antonio García Rabasco  [106.1, 28.7]           0.045668


In [10]:
shots['x'] = shots['location'].apply(lambda loc: loc[0])
shots['y'] = shots['location'].apply(lambda loc: loc[1])

GOAL_X = 120
GOAL_Y = 40

shots['shot_distance'] = ((shots['x'] - GOAL_X)**2 + (shots['y'] - GOAL_Y)**2)**0.5

print(shots['shot_distance'].describe())

count    9168.000000
mean       19.046284
std         8.685988
min         0.632456
25%        12.000417
50%        18.261709
75%        25.269300
max        74.867416
Name: shot_distance, dtype: float64


In [6]:
mean_dist    = shots['shot_distance'].mean()
median_dist  = shots['shot_distance'].median()
std_dist     = shots['shot_distance'].std()

print(f"Mean shot distance:   {mean_dist:.2f}")
print(f"Median shot distance: {median_dist:.2f}")
print(f"Std deviation:        {std_dist:.2f}")

Mean shot distance:   19.05
Median shot distance: 18.26
Std deviation:        8.69


In [7]:
shots['distance_percentile'] = shots['shot_distance'].rank(pct=True) * 100

for p in [25, 50, 75, 90]:
    val = shots['shot_distance'].quantile(p/100)
    print(f"{p}th percentile: {val:.2f}")

25th percentile: 12.00
50th percentile: 18.26
75th percentile: 25.27
90th percentile: 30.04
